# XGBoost Baseline — Clasificación binaria incendio vs megaincendio

Dataset de entrada: `data/processed/conaf_enriched_latest.parquet`  
Variable objetivo: `superficie_quemada_total_ha >= MEGAFIRE_HA_THRESHOLD`

**Antes de ejecutar las celdas de entrenamiento**, completa los placeholders en la celda de configuración.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb # Uso de librería xgb
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PROCESSED = Path("../data/processed")
DATA_MODELS = Path("../data/models")
DATA_MODELS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# ---------------------------------------------------------------
# PLACEHOLDERS — completar antes de entrenar
# ---------------------------------------------------------------
MEGAFIRE_HA_THRESHOLD = None   # TODO: definir umbral en hectáreas (ej: 500, 1000)

XGB_PARAMS = {
    "n_estimators":     None,   # TODO: número de árboles
    "max_depth":        None,   # TODO: profundidad máxima
    "learning_rate":    None,   # TODO: tasa de aprendizaje (eta)
    "subsample":        None,   # TODO: fracción de filas por árbol (0-1)
    "colsample_bytree": None,   # TODO: fracción de columnas por árbol (0-1)
    "scale_pos_weight": None,   # TODO: peso para clase positiva (calcular en celda 4)
}
# ---------------------------------------------------------------

## 1. Carga del dataset

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "conaf_enriched_latest.parquet")
print(f"{len(df):,} filas × {df.shape[1]} columnas")
df.dtypes.value_counts()

## 2. Exploración de la variable objetivo

Usar esta celda para decidir el umbral de megaincendio antes de completar `MEGAFIRE_HA_THRESHOLD`.

In [ ]:
target_col = "superficie_quemada_total_ha"
s = df[target_col].dropna()

print(s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s.clip(upper=s.quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title("Distribución (sin outliers extremos)")
axes[0].set_xlabel("ha")
np.log1p(s).hist(bins=50, ax=axes[1])
axes[1].set_title("Distribución log1p")
axes[1].set_xlabel("log(ha + 1)")
plt.tight_layout()
plt.show()

## 3. Definición de la variable objetivo

In [ ]:
assert MEGAFIRE_HA_THRESHOLD is not None, "Define MEGAFIRE_HA_THRESHOLD en la celda de configuración"

df = df.dropna(subset=[target_col]).copy()
df["es_megaincendio"] = (df[target_col] >= MEGAFIRE_HA_THRESHOLD).astype(int)

balance = df["es_megaincendio"].value_counts()
print(balance)
print(f"\nRatio negatives/positives (para scale_pos_weight): "
      f"{balance[0] / balance[1]:.2f}")

## 4. Selección de features

In [ ]:
# Columnas a excluir del training set
EXCLUDE = {
    # target y derivados
    target_col, "es_megaincendio",
    # alta cardinalidad / identificadores
    "nombre", "temporada", "datum", "geometry",
    # columnas de tiempo (se extraen features derivadas abajo)
    "fecha_hora_inicio", "fecha_inicio", "inicio", "fecha",
    "fecha_hora_inicio_utc", "hora_inicio",
    # métricas internas del join ERA5 (leakage potencial)
    "era5_dist_km", "era5_dt_hours", "era5_match_quality",
}

# Features temporales derivadas desde la columna de timestamp
ts_col = next((c for c in ("fecha_hora_inicio", "fecha_inicio", "inicio", "fecha") if c in df.columns), None)
if ts_col:
    ts = pd.to_datetime(df[ts_col], errors="coerce")
    df["mes"] = ts.dt.month
    df["hora"] = ts.dt.hour
    df["dia_del_anyo"] = ts.dt.day_of_year

# Codificación ordinal de columnas categóricas (XGBoost acepta enteros)
CAT_COLS = [c for c in ("causa", "alerta", "escenario", "region", "provincia", "comuna") if c in df.columns]
for col in CAT_COLS:
    df[col] = pd.Categorical(df[col]).codes  # -1 para NaN

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])
]

X = df[feature_cols]
y = df["es_megaincendio"]

print(f"Features: {len(feature_cols)}")
print(feature_cols)

## 5. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Positivos en train: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Positivos en test:  {y_test.sum()} ({y_test.mean():.1%})")

## 6. Entrenamiento XGBoost

In [ ]:
assert all(v is not None for v in XGB_PARAMS.values()), (
    f"Completa todos los placeholders en XGB_PARAMS: "
    f"{[k for k, v in XGB_PARAMS.items() if v is None]}"
)

model = xgb.XGBClassifier(
    **XGB_PARAMS,
    eval_metric="auc",
    random_state=RANDOM_STATE,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

## 7. Evaluación

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["incendio", "megaincendio"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["incendio", "megaincendio"],
    ax=axes[0],
)
axes[0].set_title("Matriz de confusión")

# Feature importance (top 20)
importances = pd.Series(model.feature_importances_, index=feature_cols).nlargest(20)
importances.sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Feature importance (top 20)")
axes[1].set_xlabel("Importancia")

plt.tight_layout()
plt.show()

## 8. Guardar modelo y metadata

In [ ]:
model_path = DATA_MODELS / "xgboost_baseline.json"
meta_path  = DATA_MODELS / "xgboost_baseline_meta.json"

model.save_model(model_path)

meta = {
    "threshold_ha": MEGAFIRE_HA_THRESHOLD,
    "features": list(X.columns),
    "n_features": len(feature_cols),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "auc_roc": float(auc),
    "xgb_params": XGB_PARAMS,
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print(f"Modelo guardado en: {model_path}")
print(f"Metadata guardada en: {meta_path}")